In [ ]:
import pandas as pd

df = pd.read_csv("/Users/jeremy/Documents/protests_chai/relevance/relevant_samples.csv.gz")

# Save as an ordinary CSV
#df.to_csv("/Users/jeremy/Documents/protests_chai/frames/relevant_samples.csv", index=False)

In [4]:
df.head()

,row_id,text-topic,text-topic-exp,text-entity-name,text-entity-gender,text-entity-sentiment,text-entity-sentiment-exp,text-generic-frame,text-generic-frame-exp,text-issue-frame,...,language,maintext,source_domain,url,political_leaning,month,keyword_matches,keyword_match_count,word_count,label
0,1,Ethiopia's Political Instability and State of ...,The article discusses the declaration of a sta...,Ethiopia's Amhara Region,NaN,Negative,The article discusses a state of emergency bei...,"['Security and defense', 'Economic', 'Policy p...",The article discusses a state of emergency dec...,Political Instability and Security Concerns,...,en,Ethiopia’s Council of Ministers is declaring a...,www.foxnews.com,https://www.foxnews.com/world/ethiopia-declare...,right,2023-08-01_2023-08-31,"['protesters', 'protests', 'curfews']",3,263,RELEVANT
1,5,Climate Change Protests,The article discusses a Greenpeace demonstrati...,Rishi Sunak,male,negative,The article portrays Rishi Sunak in a negative...,"['Economic', 'Environmental', 'Policy prescrip...",The article discusses the economic implication...,Climate Arsonist,...,en,Greenpeace demonstrators draped the country es...,nypost.com,https://nypost.com/2023/08/03/uk-prime-ministe...,right_lean,2023-08-01_2023-08-31,"['demonstrators', 'protest', 'protest', 'demon...",7,488,RELEVANT
2,6,Education Policy,The article discusses a family-led lawsuit aga...,"Parents from Montgomery County, Maryland",NaN,Negative,The parents are rallying against a policy they...,"['Legality, constitutionality and jurispudence...",The article discusses a family-led lawsuit aga...,Religious Freedom and Parental Rights,...,en,"Parents from Montgomery County, Maryland, rall...",www.washingtontimes.com,https://www.washingtontimes.com/news/2023/aug/...,right_lean,2023-08-01_2023-08-31,"['demonstration', 'rally', 'rally', 'rally']",4,429,RELEVANT
3,7,Tennessee School Shooting and Gun Control Prot...,The article discusses a judge's ruling to allo...,Justin Jones,male,positive,Jones is portrayed in a positive light as he a...,"['Economic', 'Capacity and resources', 'Fairne...",The article discusses a judge's ruling on allo...,Democracy and Free Speech,...,en,A judge ruled to let people bring signs back i...,www.foxnews.com,https://www.foxnews.com/politics/tennessee-jud...,right,2023-08-01_2023-08-31,"['PROTESTORS', 'protest', 'demonstrations']",3,859,RELEVANT
4,9,Economic Protests in Syria,The article discusses widespread protests in s...,Bashar Assad,male,negative,The article discusses widespread anger over th...,"['Economic', 'Capacity and resources', 'Fairne...",The article discusses the economic crisis in S...,Economic Struggle and Government Accountability,...,en,BEIRUT (AP) — Protests spread Monday in two go...,www.washingtontimes.com,https://www.washingtontimes.com/news/2023/aug/...,right_lean,2023-08-01_2023-08-31,"['Protests', 'protests', 'protests', 'uprising...",13,498,RELEVANT


In [6]:
import re
import ast
import pandas as pd

sentence_splitter = re.compile(r'(?<=[.!?])\s+')


def normalize_keyword_matches(keyword_matches):
    """
    Converts keyword_matches into a clean list of strings.
    Handles:
    - real lists
    - stringified lists: "['protest', 'protests']"
    - comma-separated strings: "protest, protests"
    """
    if keyword_matches is None:
        return []

    if isinstance(keyword_matches, float) and pd.isna(keyword_matches):
        return []

    if isinstance(keyword_matches, list):
        return [str(k).strip() for k in keyword_matches if str(k).strip()]

    if isinstance(keyword_matches, str):
        s = keyword_matches.strip()

        if not s:
            return []

        # Handles stringified lists
        if s.startswith("[") and s.endswith("]"):
            try:
                parsed = ast.literal_eval(s)
                if isinstance(parsed, list):
                    return [str(k).strip() for k in parsed if str(k).strip()]
            except Exception:
                pass

        # Fallback for comma-separated strings
        return [k.strip().strip("'").strip('"') for k in s.split(",") if k.strip()]

    return []


def extract_protest_windows_from_matches(text, keyword_matches, window_before=2, window_after=2):
    if not isinstance(text, str) or not text.strip():
        return ""

    keywords = normalize_keyword_matches(keyword_matches)

    if not keywords:
        return ""

    # Deduplicate keywords
    keywords = sorted(set(keywords), key=len, reverse=True)

    # Add word boundaries so "protest" doesn't match weird substrings
    combined_pattern = re.compile(
        r"\b(" + "|".join(re.escape(k) for k in keywords) + r")\b",
        flags=re.IGNORECASE
    )

    sentences = sentence_splitter.split(text.strip())

    windows = []

    for i, sent in enumerate(sentences):
        if combined_pattern.search(sent):
            win_start = max(0, i - window_before)
            win_end = min(len(sentences) - 1, i + window_after)
            windows.append((win_start, win_end))

    if not windows:
        return ""

    # Merge overlapping/adjacent windows
    windows.sort()
    merged = [windows[0]]

    for start, end in windows[1:]:
        last_start, last_end = merged[-1]

        if start <= last_end + 1:
            merged[-1] = (last_start, max(last_end, end))
        else:
            merged.append((start, end))

    chunks = [
        " ".join(sentences[start:end + 1])
        for start, end in merged
    ]

    return "\n\n".join(chunks)


df["protest_excerpt"] = df.apply(
    lambda row: extract_protest_windows_from_matches(
        row["maintext"],
        row["keyword_matches"],
        window_before=2,
        window_after=2
    ),
    axis=1
)

df["excerpt_length"] = df["protest_excerpt"].str.len()

print(df["excerpt_length"].describe())
print(df["protest_excerpt"].replace("", pd.NA).notna().mean())

count    9283.000000
mean     1989.877195
std      1077.759713
min       204.000000
25%      1207.000000
50%      1760.000000
75%      2518.500000
max      7569.000000
Name: excerpt_length, dtype: float64
1.0


In [7]:
print(df["maintext"].str.len().describe())

count    9283.000000
mean     3850.596251
std      1631.032818
min       691.000000
25%      2584.500000
50%      3595.000000
75%      4986.500000
max      8012.000000
Name: maintext, dtype: float64


In [8]:
df.columns

Index(['row_id', 'text-topic', 'text-topic-exp', 'text-entity-name',
       'text-entity-gender', 'text-entity-sentiment',
       'text-entity-sentiment-exp', 'text-generic-frame',
       'text-generic-frame-exp', 'text-issue-frame', 'text-issue-frame-exp',
       'uuid', 'title', 'img-generic-frame', 'img-frame-exp',
       'img-entity-name', 'img-entity-gender', 'img-entity-sentiment',
       'img-entity-sentiment-exp', 'image-url', 'gpt-topic', 'authors',
       'date_publish', 'description', 'language', 'maintext', 'source_domain',
       'url', 'political_leaning', 'month', 'keyword_matches',
       'keyword_match_count', 'word_count', 'label', 'protest_excerpt',
       'excerpt_length'],
      dtype='str')

In [9]:
df.to_csv("/Users/jeremy/Documents/protests_chai/frames/truncated_samples.csv.gz", index=False, compression="gzip")


In [12]:
import pandas as pd

df = pd.read_csv("/Users/jeremy/Documents/protests_chai/frames/truncated_samples.csv.gz")

df.head()

,row_id,text-topic,text-topic-exp,text-entity-name,text-entity-gender,text-entity-sentiment,text-entity-sentiment-exp,text-generic-frame,text-generic-frame-exp,text-issue-frame,...,source_domain,url,political_leaning,month,keyword_matches,keyword_match_count,word_count,label,protest_excerpt,excerpt_length
0,1,Ethiopia's Political Instability and State of ...,The article discusses the declaration of a sta...,Ethiopia's Amhara Region,NaN,Negative,The article discusses a state of emergency bei...,"['Security and defense', 'Economic', 'Policy p...",The article discusses a state of emergency dec...,Political Instability and Security Concerns,...,www.foxnews.com,https://www.foxnews.com/world/ethiopia-declare...,right,2023-08-01_2023-08-31,"['protesters', 'protests', 'curfews']",3,263,RELEVANT,ETHIOPIA’S AMHARA ETHNIC GROUP ACCUSES BIDEN O...,1321
1,5,Climate Change Protests,The article discusses a Greenpeace demonstrati...,Rishi Sunak,male,negative,The article portrays Rishi Sunak in a negative...,"['Economic', 'Environmental', 'Policy prescrip...",The article discusses the economic implication...,Climate Arsonist,...,nypost.com,https://nypost.com/2023/08/03/uk-prime-ministe...,right_lean,2023-08-01_2023-08-31,"['demonstrators', 'protest', 'protest', 'demon...",7,488,RELEVANT,Greenpeace demonstrators draped the country es...,2120
2,6,Education Policy,The article discusses a family-led lawsuit aga...,"Parents from Montgomery County, Maryland",NaN,Negative,The parents are rallying against a policy they...,"['Legality, constitutionality and jurispudence...",The article discusses a family-led lawsuit aga...,Religious Freedom and Parental Rights,...,www.washingtontimes.com,https://www.washingtontimes.com/news/2023/aug/...,right_lean,2023-08-01_2023-08-31,"['demonstration', 'rally', 'rally', 'rally']",4,429,RELEVANT,"Parents from Montgomery County, Maryland, rall...",1971
3,7,Tennessee School Shooting and Gun Control Prot...,The article discusses a judge's ruling to allo...,Justin Jones,male,positive,Jones is portrayed in a positive light as he a...,"['Economic', 'Capacity and resources', 'Fairne...",The article discusses a judge's ruling on allo...,Democracy and Free Speech,...,www.foxnews.com,https://www.foxnews.com/politics/tennessee-jud...,right,2023-08-01_2023-08-31,"['PROTESTORS', 'protest', 'demonstrations']",3,859,RELEVANT,"""Don’t be cowards. Have this discussion,"" Jone...",1952
4,9,Economic Protests in Syria,The article discusses widespread protests in s...,Bashar Assad,male,negative,The article discusses widespread anger over th...,"['Economic', 'Capacity and resources', 'Fairne...",The article discusses the economic crisis in S...,Economic Struggle and Government Accountability,...,www.washingtontimes.com,https://www.washingtontimes.com/news/2023/aug/...,right_lean,2023-08-01_2023-08-31,"['Protests', 'protests', 'protests', 'uprising...",13,498,RELEVANT,BEIRUT (AP) — Protests spread Monday in two go...,3131
